# 🛠️ Lab 5B — GPUs in Real AI Projects
**Foothold · GPU Computing for AI · Unit 5 (second half) · Book: Step 5, Sections 5.6–5.C**

The practical skills for real projects: inspect the software stack, measure memory, survive "CUDA out of memory", understand multi-GPU training, plan a project, and run a real language model on the GPU.

| Part | What you do | Goes with Foothold lesson |
|---|---|---|
| A | Versions, a portable device picker, real kernel names | 5.6 The Software Stack |
| B | Watch memory grow step by step; check the 16-bytes rule | 5.7 Will It Fit? |
| C | Cause an OOM, read it, fix it; catch a leak | 5.8 nvidia-smi & Out-of-Memory |
| D | Prove data parallelism on one GPU | 5.9 Many GPUs & Other Accelerators |
| E | Plan a project, then run a real language model | 5.C Grand Capstone |

⏱️ About 2 hours. 📘 Textbook: D2L Ch 13 (computational performance, multiple GPUs) · Kirk & Hwu Ch 1 and 16 · Hamacher Ch 8, §12.3–12.5. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`) saying what it does. Read them as you go.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

---
## Part A · The Software Stack  *(Unit 5.6)*

Check every layer: PyTorch, the CUDA version it was built for, cuDNN, and the driver.

In [ ]:
print("PyTorch       :", torch.__version__)   # the PyTorch layer
print("Built for CUDA:", torch.version.cuda)   # the CUDA version PyTorch was built for (None = a CPU-only build)
print("cuDNN         :", torch.backends.cudnn.version())   # NVIDIA's neural-network library
print("GPU available :", torch.cuda.is_available())   # can PyTorch actually reach a GPU?
# the driver layer: ask nvidia-smi for the GPU's name and the NVIDIA driver's version
!nvidia-smi --query-gpu=name,driver_version --format=csv

**A portable device picker:** the same code runs on NVIDIA, Apple or CPU.

In [ ]:
if torch.cuda.is_available():   # an NVIDIA GPU (or an AMD one with ROCm)?
    dev = "cuda"   # use it
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():   # an Apple M-series GPU?
    dev = "mps"   # use that
else:   # otherwise...
    dev = "cpu"   # ...the CPU
print("Using:", dev)   # show the choice

**Real kernel names:** profile a small model and look for GEMM (matrix-multiply) kernels, element-wise kernels and copies.

In [ ]:
from torch.profiler import profile, ProfilerActivity   # PyTorch's built-in profiler
net = nn.Sequential(nn.Linear(1024, 4096), nn.ReLU(), nn.Linear(4096, 10)).to(device)   # a small model on the GPU
x_cpu = torch.randn(256, 1024)   # a batch that starts on the CPU, so we also see a copy
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:   # record the CPU and the GPU
    for _ in range(5):   # five forward passes
        y = net(x_cpu.to(device))   # copy the batch over, then run the model
    sync()   # wait for the GPU before the recording stops
gpu_rows = [e for e in prof.key_averages() if getattr(e, "self_device_time_total", 0) > 0]   # the entries that ran on the GPU
for e in sorted(gpu_rows, key=lambda e: -e.self_device_time_total)[:6]:   # the 6 that took the most GPU time
    print(f"{e.key[:58]:58s} {e.self_device_time_total:7.0f} µs")   # its name and GPU time in microseconds

### ✏️ YOUR TURN (A)
1. Write down the PyTorch, CUDA and driver versions. Is the driver new enough for the CUDA build (Unit 5.6's rule)?
2. In the profiler list, find one GEMM kernel, one element-wise (ReLU) kernel and one memory copy. Copy their names here.

*Your answers:*

---
## Part B · Memory, Step by Step  *(Unit 5.7)*

A wide network (about 2 crore parameters) trained in **FP32 with Adam**: weights 4 + gradients 4 + Adam's two states 4 + 4 = **16 bytes per parameter**, plus activations. Watch `memory_allocated()` after each stage.

In [ ]:
def mem_mb():   # GPU memory currently held by live tensors, in MB
    return torch.cuda.memory_allocated() / 1e6   # bytes ÷ 1e6 = MB

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()   # start from a clean slate
stages = [("start", mem_mb())]   # a list of (stage name, memory) pairs
model = nn.Sequential(nn.Linear(784, 4096), nn.ReLU(), nn.Linear(4096, 4096), nn.ReLU(), nn.Linear(4096, 10)).to(device)   # about 2 crore parameters
n_params = sum(p.numel() for p in model.parameters())   # count them
stages.append(("model created (weights)", mem_mb()))   # weights: 4 bytes each
x = torch.randn(512, 784, device=device); y = torch.randint(0, 10, (512,), device=device)   # a made-up batch of 512
loss = nn.functional.cross_entropy(model(x), y)   # forward pass: the activations are saved for the backward pass
stages.append(("after forward (+ activations)", mem_mb()))   # measure
loss.backward()   # backward: gradients created, saved activations freed
stages.append(("after backward (+ gradients, − activations)", mem_mb()))   # measure
opt = torch.optim.Adam(model.parameters())   # the Adam optimiser...
opt.step()   # ...creates its two extra numbers per parameter on its first step
stages.append(("after Adam step (+ its two states)", mem_mb()))   # measure
for name, mb in stages:   # show every stage
    print(f"{name:45s} {mb:9.1f} MB")   # its name and memory
print(f"\nParameters: {n_params:,}")   # the parameter count
print(f"Prediction, weights + grads + Adam = 16 B/param: {16 * n_params / 1e6:.0f} MB")   # what the 16-bytes rule predicts

**Activations grow with the batch:** measure the peak memory of one forward + backward at several batch sizes.

In [ ]:
for B in [64, 256, 1024, 4096]:   # four batch sizes
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()   # reset the "highest so far" counter
    xb = torch.randn(B, 784, device=device); yb = torch.randint(0, 10, (B,), device=device)   # a made-up batch of B
    base = mem_mb()   # memory before the step
    nn.functional.cross_entropy(model(xb), yb).backward()   # one forward + backward
    peak = torch.cuda.max_memory_allocated() / 1e6   # the highest point during it
    print(f"batch {B:5d}: peak extra memory for this step ≈ {peak - base:8.1f} MB")   # the extra memory this batch needed
    model.zero_grad(set_to_none=True)   # throw the gradients away before the next size

### ✏️ YOUR TURN (B)
1. Which stage added the most memory? Did the total added match the 16-bytes-per-parameter prediction (roughly)?
2. Roughly how did the activation memory change when the batch went from 1,024 to 4,096?

*Your answers:*

---
## Part C · Out of Memory: Cause It, Read It, Fix It  *(Unit 5.8)*

We ask for a batch far too big for the T4 and **catch** the error so the notebook keeps running. (The sizes suit Colab's 15 GB T4; on a smaller GPU, halve them.)

In [ ]:
def try_batch(B, amp=False):   # try one training step with a batch of B, and catch an out-of-memory error
    torch.cuda.empty_cache()   # give back cached memory first
    try:   # attempt it; if it fails, jump to "except"
        xb = torch.randn(B, 784, device=device)   # a made-up batch of B inputs
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):   # FP16 inside this block, if amp
            h = model(xb)   # forward pass
        h.float().sum().backward()   # a simple stand-in loss, then the backward pass
        model.zero_grad(set_to_none=True)   # throw the gradients away
        print(f"batch {B:,} (amp={amp}): worked")   # success
    except torch.cuda.OutOfMemoryError as e:   # the GPU ran out of memory
        print(f"batch {B:,} (amp={amp}): OutOfMemoryError")   # say so...
        print("   " + ". ".join(str(e).split(". ")[:3]) + ".")   # ...and show the first three sentences of PyTorch's message
    finally:   # "finally" runs whether it worked or failed
        torch.cuda.empty_cache()   # clean up

try_batch(400_000)   # far too big
try_batch(100_000)   # fix 1: a smaller batch
try_batch(150_000, amp=True)   # fix 2: mixed precision about halves the activation memory

**The classic leak:** collecting model outputs during evaluation **without** `torch.no_grad()` keeps every batch's graph and activations alive.

In [ ]:
def eval_memory(use_no_grad, batches=15):   # "evaluate" 15 batches, keeping every output, and record the memory
    torch.cuda.empty_cache()   # start clean
    kept, trace = [], []   # kept = the outputs we collect; trace = memory after each batch
    for _ in range(batches):   # one batch at a time
        xb = torch.randn(2048, 784, device=device)   # a made-up batch
        if use_no_grad:   # the right way...
            with torch.no_grad():   # ...no history is built: the output is just numbers
                kept.append(model(xb))   # keep it
        else:   # the classic mistake...
            kept.append(model(xb))   # ...the history and saved activations are kept alive too!
        trace.append(mem_mb())   # memory after this batch
    del kept   # let go of everything we collected
    return trace   # the memory after each batch

leak, ok = eval_memory(False), eval_memory(True)   # run both versions
print(f"after 15 batches: without no_grad {leak[-1]:7.0f} MB   with no_grad {ok[-1]:7.0f} MB")   # compare the end points

In [ ]:
plt.figure(figsize=(6, 3.5))                                          # a new chart
plt.plot(leak, "o-", c="tab:red", label="without no_grad (leak)")     # red line: the leak
plt.plot(ok, "s-", c="tab:green", label="with torch.no_grad()")       # green line: the fix
plt.xlabel("evaluation batch"); plt.ylabel("memory allocated (MB)")   # axis labels
plt.legend(); plt.grid(alpha=0.3)                                     # colour key and faint grid
plt.title("A climbing curve = a leak"); plt.show()                    # title, then draw it

### ✏️ YOUR TURN (C)
1. Copy the first line of the OOM message. How much did it try to allocate, and how much was free?
2. How much memory did the leaking version use after 15 batches compared with the no_grad version?

*Your answers:*

---
## Part D · Prove Data Parallelism  *(Unit 5.9)*

Four GPUs each take a quarter of the batch, compute gradients, and average them (all-reduce). Here we do it with ONE device: split the batch into 4 slices, compute each slice's gradient, average, and compare with the full-batch gradient.

In [ ]:
torch.manual_seed(0)   # the same starting weights every time
net = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)   # a small network
X = torch.randn(256, 784, device=device); Y = torch.randint(0, 10, (256,), device=device)   # one batch of 256

net.zero_grad()   # one "GPU" with the whole batch: clean slopes...
nn.functional.cross_entropy(net(X), Y).backward()   # ...forward and backward on all 256
full = [p.grad.clone() for p in net.parameters()]   # save a copy of every parameter's slope

slice_grads = []   # four "GPUs", 64 examples each
for xs, ys in zip(X.chunk(4), Y.chunk(4)):   # .chunk(4) cuts the batch into 4 equal slices
    net.zero_grad()   # each "GPU" starts with clean slopes
    nn.functional.cross_entropy(net(xs), ys).backward()   # forward and backward on its slice
    slice_grads.append([p.grad.clone() for p in net.parameters()])   # keep its slopes
averaged = [sum(g[i] for g in slice_grads) / 4 for i in range(len(full))]   # the all-reduce: add the 4 slices' slopes, divide by 4

diff = max((a - b).abs().max().item() for a, b in zip(full, averaged))   # the biggest difference anywhere
print(f"Largest difference between full-batch and averaged gradients: {diff:.2e}")   # show it
print("Identical (up to rounding): data parallelism computes the SAME update." if diff < 1e-5 else "Check the code.")   # the verdict

### ✏️ YOUR TURN (D)
1. Why does averaging work only when the slices are the same size? (Hint: what does `cross_entropy` do by default: sum or mean?)
2. In real data parallelism the averaging happens over NVLink. Using Amdahl's law, why must that exchange be fast?

*Your answers:*

---
## Part E · Plan a Project, Then Run a Real Language Model  *(Unit 5.C)*

First the planner from the capstone (prices are **assumptions**; check current rates).

In [ ]:
# each GPU: (memory in GB, FP16 peak FLOPS, memory bandwidth in bytes/s, assumed ₹ per hour)
GPUS = {"T4": (16, 65e12, 320e9, 35), "L4": (24, 121e12, 300e9, 70), "A100-40": (40, 312e12, 1555e9, 250), "A100-80": (80, 312e12, 2000e9, 350)}

def plan_train(name, params, tokens):   # plan a training job: fit, time, cost
    need = params * 18 / 1e9 + 0.5   # FIT: about 16 bytes per parameter + activations ≈ 18, plus 0.5 GB overhead
    print(f"\n{name}: training needs ≈ {need:.1f} GB")   # show the memory needed
    for g, (mem, tf, bw, price) in GPUS.items():   # for each kind of GPU
        n = max(1, int(-(-need // mem)))   # how many GPUs to hold it (round up, at least 1)
        hours = 6 * params * tokens / (tf * 0.35) / 3600 / n   # TIME: 6 × parameters × tokens ÷ (peak × 35%), shared over n GPUs
        print(f"   {g:8s} {n} GPU(s)  {hours:8.2f} h  ≈ ₹{hours * n * price:,.0f}")   # COST: hours × GPUs × price

def plan_chat(name, params, answers, bytes_per=2):   # plan a chatbot: fit, time, cost per day
    need = params * bytes_per * 1.2 / 1e9 + 0.5   # FIT: the weights (+20% for activations and cache) + overhead
    print(f"\n{name}: running needs ≈ {need:.1f} GB")   # show the memory needed
    for g, (mem, tf, bw, price) in GPUS.items():   # for each kind of GPU
        n = max(1, int(-(-need // mem)))   # how many GPUs to hold it
        hours = answers * 270 * (params * bytes_per / (bw * 0.6)) / 8 / 3600 / n   # TIME: each token reads all the weights at 60% of bandwidth; 270 tokens per answer; 8 answers at a time
        print(f"   {g:8s} {n} GPU(s)  {hours:8.2f} GPU-h/day  ≈ ₹{hours * n * price:,.0f}/day")   # COST per day

plan_chat("Helpdesk chatbot, 7B, FP16", 7e9, 20_000)   # 20,000 answers a day, 2 bytes per weight
plan_chat("Helpdesk chatbot, 7B, INT8", 7e9, 20_000, bytes_per=1)   # the same, 1 byte per weight
plan_train("Fine-tune a 1B model on 50M tokens", 1e9, 50e6)   # a fine-tuning job

**A real language model.** `distilgpt2` has 82 million parameters (download ≈ 350 MB). Generate 100 tokens on the CPU and on the GPU.
Prediction (Unit 5.3): each token reads all the weights, so on the T4 in FP16 the time per token is at least 82M × 2 bytes ÷ 320 GB/s ≈ 0.5 ms (≈ 2,000 tokens/s at most).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM   # Hugging Face's library of ready-trained models
name = "distilgpt2"   # a small real language model: 82 million parameters
tok = AutoTokenizer.from_pretrained(name)   # its tokenizer: text → token IDs and back (Step 1.2)
prompt = "The GPU is fast because"   # the text the model will continue
lm_speed = {}   # device → tokens per second
for dev in ["cpu", "cuda"]:   # on the CPU, then on the GPU
    dtype = torch.float16 if dev == "cuda" else torch.float32   # FP16 on the GPU, FP32 on the CPU
    lm = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).to(dev)   # download (the first time) and load the model
    ids = tok(prompt, return_tensors="pt").to(dev)   # the prompt as token IDs, on the same device
    lm.generate(**ids, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)   # a warm-up: 5 tokens
    if dev == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()   # wait, and reset the memory counter
    t0 = time.perf_counter()   # start the stopwatch
    out = lm.generate(**ids, max_new_tokens=100, min_new_tokens=100, do_sample=False, pad_token_id=tok.eos_token_id)   # exactly 100 new tokens, always the most likely one
    if dev == "cuda": torch.cuda.synchronize()   # wait for the GPU before stopping the clock
    lm_speed[dev] = 100 / (time.perf_counter() - t0)   # speed = 100 tokens ÷ seconds
    mem = f", peak memory {torch.cuda.max_memory_allocated() / 1e6:.0f} MB" if dev == "cuda" else ""   # GPU memory used
    print(f"{dev.upper():4s} ({str(dtype)[6:]}): {lm_speed[dev]:6.0f} tokens/s{mem}")   # show the speed
print("\nSample output:", tok.decode(out[0], skip_special_tokens=True)[:260])   # the generated text, as words

Always picking the single most likely next token (`do_sample=False`) makes small models repeat themselves; real chatbots sample with a little randomness.

### ✏️ YOUR TURN (E) — record your results
| | CPU | GPU (T4, FP16) |
|---|---|---|
| Tokens per second | | |
| Peak GPU memory | — | |
| Bandwidth limit (predicted) | — | ≈ 2,000 tokens/s |

1. How far below the bandwidth limit was the real GPU speed? Explain using Amdahl's law and launch overheads (Units 2.8, 4.4).
2. Using the planner, which GPU would you recommend for the helpdesk chatbot, and why? State your assumptions.

*Your answers:*

---
## 🎓 Course wrap-up
You can now explain and measure every link in the chain: **AI = multiply-adds → GPUs do them in parallel → the CPU feeds the GPU over PCIe → speed comes from reuse, low precision and good feeding → real projects are planned by memory, time and cost.**

**Next:** your mini-project. Open `MiniProject_Template.ipynb` from the course site.